# Prepare metrics

This notebook follows the shared diagnostic workflow.


## 1. Libraries


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr


## 2. User setup and project configuration


In [ ]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# Edit these three paths to relocate inputs, the analysis checkout, or outputs.
# =============================================================================
from pathlib import Path
import os
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
WORKFLOW_NAME = "analysis_lac"


In [ ]:
# Derived paths: built from the parameters above.
import sys
from pathlib import Path

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
)


In [ ]:
# === Notebook path configuration ===
from pathlib import Path
import sys

CWD = WORK_DIR
DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"

if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

# === Core Utilities ===
import os
import logging
from collections import defaultdict
from typing import Tuple, List, Dict, Optional

# === Date/Time Handling ===
from datetime import datetime
# import datetime as dt  # Optional: alias if dt.datetime(...) is preferred

# === Numerical and Data Tools ===
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xcdat as xc
import xskillscore as xs

import warnings

from util.lnd_atm_model_data import ModelDataReader

from util.lnd_atm_observation_data import ObsDataReader

from util.lnd_atm_feedback import (
    LandAtmosphereFeedbackAnalyzer, 
    plot_lag_with_ci, 
    plot_feedback_map_with_sig
)
from util.dask_helpers import (
    configure_dask,
    open_dataset_lazy,
    persist_if_dask,
)
from configs.output_paths import workflow_output_dirs


## 3. Processing, cache, and plotting utilities


## 4. Run the workflow and show results


In [ ]:
if __name__ == "__main__":
    top_path = str(INPUT_DATA_ROOT)
    data_dir, figure_dir = workflow_output_dirs("analysis_lac", "feedback", output_root=DIAGNOSTIC_OUTPUT_ROOT)
    out_path = str(data_dir)
    fig_path = str(figure_dir)
    dask_client = configure_dask(use_distributed=False)

    # === Configuration and inputs ===
    exp_info = get_dart_config()
    exp_dict = exp_info['experiments'] 
    exp_cfg  = exp_info['global']
    
    reg_dict = {
        'NH':      'Northern Hemisphere', 
        'SH':      'Southern Hemisphere', 
        'Tropics': 'Tropics', 
        'NA':      'North America',
        'Global':  'global'
    }
    

    exp_list = ["CTRL10", "ERA510", "DART20", "DART40"]
    exp_path_base = out_path
    land_mask = open_dataset_lazy(f"{top_path}/reference/lnd_sea_mask/landmask_1x1.nc")["landfrac"] > 0.5
    land_mask = persist_if_dask(land_mask)

    feedback_analyzer = LandAtmosphereFeedbackAnalyzer(land_mask=land_mask)

    for exp in exp_list:
        print(f"Processing {exp}")
        exp_dir = os.path.join(exp_path_base, exp)
        members = load_member_data(exp_dir, var_list=["SMOIS", "PRECT"], use_dask=True)

        model_dict = {exp: members}
        results = feedback_analyzer.compute_lag_correlation_curve(
            model_dict, var_land="SMOIS", var_atm="PRECT", lags=list(range(-48, 49, 6))
        )
        
        feedback_analyzer.save_to_netcdf(results, os.path.join(out_path, f"lagcorr_{exp}_SMOIS_PRECT.nc"))
    
    # Initialize
    analyzer = LandAtmosphereFeedbackAnalyzer(
        land_mask=land_frac > 0.5
    )

    # Example 1: Lag correlation curve (SMOIS → PRECT)
    curve = analyzer.compute_lag_correlation_curve(
        data_dict, "SMOIS", "PRECT", 
        lags=list(range(-48, 49, 6))
    )
    
    lag_fig, _ = plot_lag_with_ci(curve, title="SMOIS → PRECT Feedback")
    lag_fig.savefig(os.path.join(fig_path, "land_feedback_SMOIS_PRECT.pdf"), bbox_inches="tight")

    # Example 2: Spatial map (TS → LHFLX at +24h)
    rmap = analyzer.compute_spatial_map(data_dict["DART20"], "TS", "LHFLX", lag=24)
    map_fig, _ = plot_feedback_map_with_sig(rmap, title="TS → LHFLX Spatial Feedback")
    map_fig.savefig(os.path.join(fig_path, "land_feedback_TS_LHFLX_map.pdf"), bbox_inches="tight")

    # Example 3: Save curve to NetCDF
    analyzer.save_to_netcdf(curve, os.path.join(out_path, "land_feedback_SMOIS_PRECT.nc"))
